For each language:
1. Takes the first N_SENTENCES lines
2. Applies preprocessing (cleaning + lowercase)
3. Tokenizes
4. Saves the preprocessed text in texts_preprocessed/
5. Saves the TSV files in output_preprocessed/

In [21]:
import re
import unicodedata
from collections import Counter
from pathlib import Path

INPUT_DIR        = Path("texts")
TEXT_OUTPUT_DIR  = Path("texts_preprocessed")   # clean texts
TSV_OUTPUT_DIR   = Path("output_preprocessed")  # frequency TSVs
N_SENTENCES      = 20000

TEXT_OUTPUT_DIR.mkdir(exist_ok=True)
TSV_OUTPUT_DIR.mkdir(exist_ok=True)

LANGUAGES = {
    "el": {"language": "Greek",     "family": "IE-Hellenic",  "spacy_model": "el_core_news_sm"},
    "en": {"language": "English",   "family": "IE-Germanic",  "spacy_model": "en_core_web_sm"},
    "es": {"language": "Spanish",   "family": "IE-Romance",   "spacy_model": "es_core_news_sm"},
    "fi": {"language": "Finnish",   "family": "Uralic",       "spacy_model": "fi_core_news_sm"},
    "hu": {"language": "Hungarian", "family": "Uralic",       "spacy_model": None},
    "it": {"language": "Italian",   "family": "IE-Romance",   "spacy_model": "it_core_news_sm"},
    "mt": {"language": "Maltese",   "family": "Semitic",      "spacy_model": None},
    "pl": {"language": "Polish",    "family": "IE-Slavic",    "spacy_model": "pl_core_news_sm"},
}


In [22]:
JUSTIFICATION_HEADERS = {
    "justification", "begründung", "justificación", "justificazione",
    "motivering", "uzasadnienie", "indoklás", "αιτιολόγηση",
    "ġustifikazzjoni", "odůvodnění", "põhjendus", "pagrindimas",
    "pamatojums", "zdůvodnění", "utemeljitev", "perustelut",
    "motivação", "motivazione", "motivación", "justificare",
    "odôvodnenie", "motivuojamoji",
}

AMENDMENT_HEADERS = {
    # Romance
    "enmienda", "emenda", "emendement", "modifica", "modificare",
    # Germanic
    "amendment", "änderungsantrag", "ändringsförslag", "ændring",
    # Slavic
    "pozměňovací návrh", "zmiana", "pozmeňujúci", "sprememba",
    # Uralic
    "módosítás", "tarkistus", "muudatusettepanek", "muudatus",
    # Baltic
    "pakeitimas", "grozījums",
    # Greek
    "τροπολογία", "τροπολογίες", "τροποποιήσεις",
    # Maltese
    "emendament",
}

# Lines indicating original language of the MEP — always skip
ORIGINAL_LANG_PATTERN = re.compile(r'^or\.\s+\w+$', re.IGNORECASE)


def is_metadata_line(line: str) -> bool:
    """Detects lines that are only metadata and should be removed."""
    # Very short lines (codes, amendment numbers)
    if len(line) < 4:
        return True
    # Lines that are all uppercase (FINAL, INDEX, PROCEDURE)
    if line.isupper() and len(line.split()) <= 3:
        return True
    # Lines starting with codes like A6-, PE
    if re.match(r'^(A\d|B\d|PE\s|COM|DO\s)', line):
        return True
    # Voting lines — Latin scripts
    if re.match(r'^(a favor|en contra|abstenciones|for:|against:|in favour)', line.lower()):
        return True
    # Voting lines — Greek
    if re.match(r'^(υπέρ:|κατά:|αποχές:)', line.lower()):
        return True
    # Original language indicator: "Or. en", "Or. fr", etc.
    if ORIGINAL_LANG_PATTERN.match(line.strip()):
        return True
    return False

# ── Preprocessing ──────────────────────────────────────────────────────────────

def truncate_by_sentences(text: str, n: int) -> str:
    """Takes the first n non-empty lines."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return "\n".join(lines[:n])


def clean_line(line: str) -> str:
    # FIRST: replace / with space to prevent word fusion
    line = line.replace('/', ' ')

    # Remove full URLs
    line = re.sub(r'https?://\S+', '', line)
    line = re.sub(r'www\.\S+', '', line)

    # Remove legislative codes like A6-0012 2005, C6‑0188 2004
    line = re.sub(r'\b[A-Z]\d[\-‑][0-9]+\s[0-9]+\b', '', line)

    # Remove references like COM(2003)0741, P5_TA(2004)0301
    line = re.sub(r'\b[A-Z0-9_]+\([0-9]+\)[0-9]+\b', '', line)

    # Remove procedural codes: COD, AVC, CNS, APP, INI, RSP, IMM, BUD, etc.
    line = re.sub(r'\b(COD|AVC|CNS|APP|INI|RSP|IMM|BUD|REG|DIR|DEC)\b', '', line)

    # Remove dates like 26.1.2005, 1.1.2009
    line = re.sub(r'\b\d{1,2}\.\d{1,2}\.\d{4}\b', '', line)

    # Remove sequences of only numbers and punctuation
    line = re.sub(r'\b[\d\.\-\/\(\)]+\b', '', line)

    # Remove parliamentary symbols (* ** ***)
    line = re.sub(r'\*+[I]+', '', line)
    line = re.sub(r'\*+', '', line)

    # Remove isolated long dashes
    line = re.sub(r'\s[–‑—]\s', ' ', line)

    # Clean multiple spaces
    line = re.sub(r'\s+', ' ', line).strip()

    return line

def filter_justifications(lines: list[str]) -> list[str]:
    """
    Removes justification sections written in foreign languages.
    Structure in DCEP documents:
        [Main text in document language]
        Justification / Begründung / Justificación / ...
        [Text in the MEP's own language — often different]
        Amendment / Enmienda / ...   ← resets the flag
    """
    filtered = []
    in_justification = False

    for line in lines:
        line_lower = line.strip().lower()

        # New amendment header → reset flag (skip the header line itself)
        if any(line_lower.startswith(h) for h in AMENDMENT_HEADERS):
            in_justification = False
            continue

        # Justification header → activate flag (skip the header line itself)
        if line_lower in JUSTIFICATION_HEADERS:
            in_justification = True
            continue

        if in_justification:
            continue

        filtered.append(line)

    return filtered


def preprocess_text(text: str, n: int) -> str:
    """Full preprocessing pipeline."""
    # 1. Truncate to N lines
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    lines = lines[:n]

    # 2. Filter metadata lines
    lines = [l for l in lines if not is_metadata_line(l)]

    # 3. Remove justification sections in foreign languages
    lines = filter_justifications(lines)

    # 4. Clean each line
    lines = [clean_line(l) for l in lines]

    # 5. Remove lines that became empty after cleaning
    lines = [l for l in lines if len(l.strip()) > 3]

    return "\n".join(lines)


# ── Tokenization ───────────────────────────────────────────────────────────────

def is_valid_token(token: str) -> bool:
    return any(unicodedata.category(c).startswith("L") for c in token)


def is_valid_word(word: str) -> bool:
    if word.startswith("http") or word.startswith("www") or "://" in word:
        return False
    if len(word) > 20:          # reduced from 30 to catch merged words
        return False
    if len(word) < 2:
        return False
    if word.isdigit():
        return False
    if re.match(r'^[\d\.\-\/\(\)]+$', word):
        return False
    if re.match(r'^[A-Z]\d[\-\/]', word):
        return False
    return True


def tokenize_spacy(text: str, model_name: str):
    import spacy
    try:
        nlp = spacy.load(model_name, disable=["parser", "ner", "lemmatizer"])
        lines = text.splitlines()
        chunk_size = 10_000
        all_tokens = []
        for i in range(0, len(lines), chunk_size):
            chunk = "\n".join(lines[i:i+chunk_size])
            if not chunk.strip():
                continue
            nlp.max_length = len(chunk) + 100
            doc = nlp(chunk)
            all_tokens.extend([t.text for t in doc if is_valid_token(t.text)])
            print(f"    Batch {i//chunk_size + 1}/{(len(lines)-1)//chunk_size + 1} "
                  f"({len(all_tokens):,} tokens)", end="\r")
        print()
        return all_tokens
    except OSError:
        print(f"    [!] Model '{model_name}' not found → using regex tokenizer")
        return None


def tokenize_regex(text: str) -> list[str]:
    tokens = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    return [t for t in tokens if is_valid_token(t)]


def tokenize(text: str, lang_info: dict) -> list[str]:
    model = lang_info.get("spacy_model")
    if model:
        tokens = tokenize_spacy(text, model)
        if tokens is not None:
            return tokens
    return tokenize_regex(text)


def compute_freq_table(tokens: list[str]) -> list[tuple]:
    freq = Counter(tokens)
    rows = [(word, count, len(word))
            for word, count in freq.items()
            if is_valid_word(word)]
    rows.sort(key=lambda x: -x[1])
    return rows


def save_tsv(rows: list[tuple], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("word_form\tfrequency\tlength\n")
        for word, freq, length in rows:
            f.write(f"{word}\t{freq}\t{length}\n")

In [ ]:
'''
def truncate_by_sentences(text: str, n: int) -> str:
    """Takes the first n non-empty lines."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return "\n".join(lines[:n])

def clean_line(line: str) -> str:
    """
    Cleans a line of text:
    - Removes URLs
    - Removes legislative codes (A6-0012/2005, C6-0188/2004, etc.)
    - Removes numerical references (3820/85, COM(2003)0741, etc.)
    - Removes metadata-only lines (FINAL, ***II, PE 350.091, etc.)
    """
    # Remove full URLs
    line = re.sub(r'https?://\S+', '', line)
    line = re.sub(r'www\.\S+', '', line)

    # Remove legislative codes like A6-0012/2005, C6‑0188/2004
    line = re.sub(r'\b[A-Z]\d[\-‑][0-9]+\/[0-9]+\b', '', line)

    # Remove references like COM(2003)0741, P5_TA(2004)0301
    line = re.sub(r'\b[A-Z0-9_]+\([0-9]+\)[0-9]+\b', '', line)

    # Remove article numbers like 3820/85, 1059/2003
    line = re.sub(r'\b\d+\/\d+\b', '', line)

    # Remove dates like 26.1.2005, 1.1.2009
    line = re.sub(r'\b\d{1,2}\.\d{1,2}\.\d{4}\b', '', line)

    # Remove sequences of only numbers and punctuation
    line = re.sub(r'\b[\d\.\-\/\(\)]+\b', '', line)

    # Remove parliamentary symbols (* ** ***)
    line = re.sub(r'\*+[I]+', '', line)
    line = re.sub(r'\*+', '', line)

    # Remove isolated long dashes
    line = re.sub(r'\s[–‑—]\s', ' ', line)

    # Clean multiple spaces
    line = re.sub(r'\s+', ' ', line).strip()

    return line

def is_metadata_line(line: str) -> bool:
    """Detects lines that are only metadata and should be removed."""
    # Very short lines (codes, amendment numbers)
    if len(line) < 4:
        return True
    # Lines that are all uppercase (FINAL, INDEX, PROCEDURE)
    if line.isupper() and len(line.split()) <= 3:
        return True
    # Lines starting with codes like A6-, PE
    if re.match(r'^(A\d|B\d|PE\s|COM|DO\s)', line):
        return True
    # Voting lines (in favour:, against:)
    if re.match(r'^(a favor|en contra|abstenciones|for:|against:|in favour)', line.lower()):
        return True
    return False

def preprocess_text(text: str, n: int) -> str:
    """Full preprocessing pipeline."""
    # 1. Truncate to N lines
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    lines = lines[:n]

    # 2. Filter metadata lines
    lines = [l for l in lines if not is_metadata_line(l)]

    # 3. Clean each line
    lines = [clean_line(l) for l in lines]

    # 4. Remove lines that became empty after cleaning
    lines = [l for l in lines if len(l.strip()) > 3]

    return "\n".join(lines)

#Tokenization
def is_valid_token(token: str) -> bool:
    return any(unicodedata.category(c).startswith("L") for c in token)

def is_valid_word(word: str) -> bool:
    if word.startswith("http") or word.startswith("www") or "://" in word:
        return False
    if len(word) > 30:
        return False
    if len(word) < 2:
        return False
    if word.isdigit():
        return False
    if re.match(r'^[\d\.\-\/\(\)]+$', word):
        return False
    if re.match(r'^[A-Z]\d[\-\/]', word):
        return False
    return True

def tokenize_spacy(text: str, model_name: str):
    import spacy
    try:
        nlp = spacy.load(model_name, disable=["parser", "ner", "lemmatizer"])
        lines = text.splitlines()
        chunk_size = 10_000
        all_tokens = []
        for i in range(0, len(lines), chunk_size):
            chunk = "\n".join(lines[i:i+chunk_size])
            if not chunk.strip():
                continue
            nlp.max_length = len(chunk) + 100
            doc = nlp(chunk)
            all_tokens.extend([t.text for t in doc if is_valid_token(t.text)])
            print(f"    Batch {i//chunk_size + 1}/{(len(lines)-1)//chunk_size + 1} "
                  f"({len(all_tokens):,} tokens)", end="\r")
        print()
        return all_tokens
    except OSError:
        print(f"    [!] Model '{model_name}' not found → using regex tokenizer")
        return None

def tokenize_regex(text: str) -> list[str]:
    tokens = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    return [t for t in tokens if is_valid_token(t)]

def tokenize(text: str, lang_info: dict) -> list[str]:
    model = lang_info.get("spacy_model")
    if model:
        tokens = tokenize_spacy(text, model)
        if tokens is not None:
            return tokens
    return tokenize_regex(text)

def compute_freq_table(tokens: list[str]) -> list[tuple]:
    freq = Counter(tokens)
    rows = [(word, count, len(word))
            for word, count in freq.items()
            if is_valid_word(word)]
    rows.sort(key=lambda x: -x[1])
    return rows

def save_tsv(rows: list[tuple], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("word_form\tfrequency\tlength\n")
        for word, freq, length in rows:
            f.write(f"{word}\t{freq}\t{length}\n")
'''

In [23]:
summary = []

for code, info in LANGUAGES.items():
    input_path = INPUT_DIR / f"{code}.txt"
    text_out   = TEXT_OUTPUT_DIR / f"{code}_preprocessed.txt"
    tsv_out    = TSV_OUTPUT_DIR  / f"{code}_freq.tsv"

    print(f"\n[{info['language']}] ({info['family']})")

    if not input_path.exists():
        print(f"  [!] File not found: {input_path} — skipping")
        continue

    # Read original text
    full_text = input_path.read_text(encoding="utf-8")
    n_lines_original = len([l for l in full_text.splitlines() if l.strip()])
    print(f"  Original lines     : {n_lines_original:>8,}")

    # Preprocess
    clean_text = preprocess_text(full_text, N_SENTENCES)
    n_lines_clean = len([l for l in clean_text.splitlines() if l.strip()])
    print(f"  Lines after cleaning: {n_lines_clean:>7,}")
    print(f"  Characters          : {len(clean_text):>7,}")

    # Save preprocessed text
    text_out.write_text(clean_text, encoding="utf-8")
    print(f"  → Text saved to {text_out}")

    # Convert to lowercase BEFORE tokenizing
    clean_text_lower = clean_text.lower()

    # Tokenize
    tokens = tokenize(clean_text_lower, info)
    types  = set(tokens)
    print(f"  Tokens             : {len(tokens):>8,}")
    print(f"  Types              : {len(types):>8,}")

    # Save TSV
    rows = compute_freq_table(tokens)
    save_tsv(rows, tsv_out)
    print(f"  → TSV saved to {tsv_out}")

    summary.append({
        "code": code, "language": info["language"],
        "family": info["family"],
        "tokens": len(tokens), "types": len(types)
    })


[Greek] (IE-Hellenic)
  Original lines     :   46,589
  Lines after cleaning:  10,276
  Characters          : 1,542,651
  → Text saved to texts_preprocessed\el_preprocessed.txt
    Batch 2/2 (216,738 tokens)
  Tokens             :  216,738
  Types              :   14,535
  → TSV saved to output_preprocessed\el_freq.tsv

[English] (IE-Germanic)
  Original lines     :   49,917
  Lines after cleaning:  10,370
  Characters          : 1,369,100
  → Text saved to texts_preprocessed\en_preprocessed.txt
    Batch 2/2 (209,607 tokens)
  Tokens             :  209,607
  Types              :    8,529
  → TSV saved to output_preprocessed\en_freq.tsv

[Spanish] (IE-Romance)
  Original lines     :   50,000
  Lines after cleaning:  10,775
  Characters          : 1,547,248
  → Text saved to texts_preprocessed\es_preprocessed.txt
    Batch 2/2 (237,288 tokens)
  Tokens             :  237,288
  Types              :   10,818
  → TSV saved to output_preprocessed\es_freq.tsv

[Finnish] (Uralic)
  Original 

In [19]:
print("\n" + "=" * 60)
print(f"{'Language':<12} {'Family':<15} {'Tokens':>10} {'Types':>8}")
print("-" * 60)
for r in summary:
    print(f"  {r['language']:<12} {r['family']:<15} "
          f"{r['tokens']:>10,} {r['types']:>8,}")
print("=" * 60)


Language     Family              Tokens    Types
------------------------------------------------------------
  Greek        IE-Hellenic        216,825   14,538
  English      IE-Germanic        209,694    8,533
  Spanish      IE-Romance         237,389   10,822
  Finnish      Uralic             194,542   27,375
  Hungarian    Uralic             251,435   29,744
  Italian      IE-Romance         229,356   11,296
  Maltese      Semitic            182,850   18,208
  Polish       IE-Slavic           83,453   12,996


In [13]:
# Celda de diagnóstico
path = Path("texts/el.txt")
with open(path, encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"Non-empty lines: {sum(1 for l in lines if l.strip())}")
print("\nFirst 10 lines:")
for l in lines[:10]:
    print(repr(l))

Total lines: 46589
Non-empty lines: 46589

First 10 lines:
'ΤΕΛΙΚΟ\n'
'A6-0012/2005\n'
'26.1.2005\n'
'***II\n'
'ΣΥΣΤΑΣΗ ΓΙΑ ΤΗ ΔΕΥΤΕΡΗ ΑΝΑΓΝΩΣΗ\n'
'σχετικά με την κοινή θέση του Συμβουλίου ενόψει της έγκρισης κανονισμού του Ευρωπαϊκού Κοινοβουλίου και του Συμβουλίου που αφορά τους όρους πρόσβασης στα δίκτυα μεταφοράς φυσικού αερίου\n'
'(11652/2/2004 – C6‑0188/2004 – 2003/0302(COD))\n'
'Επιτροπή Βιομηχανίας, Έρευνας και Ενέργειας\n'
'Εισηγητής:\n'
'Esko Seppänen\n'


In [14]:
with open(path, encoding="utf-8") as f:
    text = f.read()

lines = [l.strip() for l in text.splitlines() if l.strip()]
print(f"After truncation:      {len(lines)}")

lines2 = [l for l in lines if not is_metadata_line(l)]
print(f"After metadata filter: {len(lines2)}")

lines3 = filter_justifications(lines2)
print(f"After justif. filter:  {len(lines3)}")

lines4 = [clean_line(l) for l in lines3]
lines4 = [l for l in lines4 if len(l.strip()) > 3]
print(f"After clean_line:      {len(lines4)}")

After truncation:      46589
After metadata filter: 39342
After justif. filter:  51
After clean_line:      50


In [15]:
with open(Path("texts/el.txt"), encoding="utf-8") as f:
    text = f.read()

lines = [l.strip() for l in text.splitlines() if l.strip()]
lines = lines[:10000]
lines = [l for l in lines if not is_metadata_line(l)]

# Ver qué líneas cortas aparecen (posibles headers)
short_lines = [l for l in lines if len(l.split()) <= 3]
from collections import Counter
print(Counter(short_lines).most_common(30))

[('Τροπολογία', 609), ('Αιτιολόγηση', 528), ('διαγράφεται', 44), ('Τίτλος', 37), ('Εξέταση στην επιτροπή', 36), ('Έγγραφα αναφοράς', 34), ('Or. en', 34), ('υπέρ:', 27), ('κατά:', 27), ('αποχές:', 26), ('Ημερομηνία έγκρισης', 25), ('Τροπολογίες του Κοινοβουλίου', 19), ('Παρατηρήσεις', 18), ('Τροποποιήσεις του Κοινοβουλίου', 17), ('Εισηγητής:', 16), ('Νομική βάση', 16), ('Διαδικαστική βάση', 16), ('Εισηγητής(ές) Ημερομηνία ορισμού', 15), ('Διαγράφεται', 15), ('22.11.2005', 15), ('5.10.2005', 13), ('28.11.2005', 11), ('Αιτιολογική σκέψη 6', 10), ('Or. fr', 10), ('άρθρο 51', 9), ('Ενισχυμένη συνεργασία', 9), ('Αιτιολογική σκέψη 7', 8), ('Εισηγητής(ές) που αντικαταστάθηκε(καν)', 8), ('Άρθρο 4', 7), ('21.6.2005', 7)]
